# 5-Fold ConvNeXt Ensembling Pipeline for Google Colab
Цей блокнот:
1. Клонує твій репозиторій GitHub з усіма скриптами, валідацією та фолдами.
2. Підключає Google Drive для доступу до `data.zip` (або `cats-and-dogs.zip`).
3. Тренує моделі на фолдах (Fold 0, 1, 2...).
4. Збирає усереднені передбачення ансамблю (OOF & Test Inference).
5. Зберігає фінальний покращений `submission_5fold_ensemble.csv`.

In [ ]:
# 1. Монтування Google Drive та розпаковка даних
import os, zipfile
from google.colab import drive

drive.mount('/content/drive')

# Шукаємо архів у Drive
drive_search_paths = ['/content/drive/MyDrive/DL', '/content/drive/MyDrive']
zip_path = None

for p in drive_search_paths:
    if os.path.exists(p):
        for f in os.listdir(p):
            if f.endswith('.zip') and any(k in f.lower() for k in ['cat', 'dog', 'pet', 'data', 'assignment']):
                zip_path = os.path.join(p, f)
                break
    if zip_path: break

print(f"Знайдено архів: {zip_path}")
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/data')

print("Дані успішно розпаковано в /content/data!")

In [ ]:
# 2. Клонуємо актуальний код з GitHub та встановлюємо бібліотеки
!rm -rf /content/repo
!git clone https://github.com/grenkka/Assignment_02_Deep_Learning.git /content/repo
!pip install timm albumentations iterative-stratification -q

# Копіюємо валідаційні спліти
!mkdir -p /content/validation
!cp /content/repo/validation/train_folds.csv /content/validation/train_folds.csv
print("Репозиторій та спліти готові!")

In [ ]:
# 3. Повний пайплайн навчання 5-Fold ансамблю ConvNeXt-Tiny
import os, random, gc
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

class CFG:
    model_name = 'convnext_tiny'
    img_size = 384
    batch_size = 64
    epochs = 6
    lr = 1e-4
    weight_decay = 1e-2
    num_classes = 5
    # Можна натренувати 3 або всі 5 фолдів
    folds_to_train = [0, 1, 2]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    folds_csv = '/content/validation/train_folds.csv'
    train_img_dir = '/content/data/train/'
    test_img_dir = '/content/data/test/'
    sample_sub = '/content/data/sample_submission.csv'

LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']

def get_transforms(mode='train'):
    if mode == 'train':
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

class PetDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        labels = torch.tensor(row[LABEL_COLS].values.astype(np.float32))
        return img, labels

def find_optimal_thresholds(y_true, y_pred_probs, steps=100):
    num_classes = y_true.shape[1]
    best_thresholds = np.full(num_classes, 0.5)
    threshold_candidates = np.linspace(0.05, 0.95, steps)
    for c in range(num_classes):
        best_c_th = 0.5
        best_c_f1 = -1.0
        for th in threshold_candidates:
            preds_c = (y_pred_probs[:, c] >= th).astype(int)
            score = f1_score(y_true[:, c], preds_c, zero_division=0)
            if score > best_c_f1:
                best_c_f1 = score
                best_c_th = th
        best_thresholds[c] = best_c_th
    return best_thresholds

def apply_postprocessing(preds):
    preds[:, 0] = np.logical_or(preds[:, 0], preds[:, 2]).astype(int)
    preds[:, 0] = np.logical_or(preds[:, 0], preds[:, 3]).astype(int)
    preds[:, 1] = np.logical_or(preds[:, 1], preds[:, 4]).astype(int)
    return preds

print("Пайплайн налаштовано! Готові до запуску фолдів.")


In [ ]:
# 4. Тренуємо фолди та зберігаємо ваги кожного
df_all = pd.read_csv(CFG.folds_csv)

trained_checkpoints = []

for fold in CFG.folds_to_train:
    save_path = f"/content/{CFG.model_name}_fold{fold}_best.pth"
    # Якщо чекпоінт для Fold 0 уже завантажений з GitHub, використовуємо його
    repo_ckpt = f"/content/repo/{CFG.model_name}_fold{fold}_best.pth"
    if fold == 0 and os.path.exists(repo_ckpt):
        print(f"=> Fold {fold} вже натреновано! Копіюю готовий чекпоінт з репозиторію.")
        !cp {repo_ckpt} {save_path}
        trained_checkpoints.append(save_path)
        continue

    print(f"\n{'='*30}\n  ТРЕНУВАННЯ FOLD {fold}\n{'='*30}")
    train_df = df_all[df_all['fold'] != fold].reset_index(drop=True)
    val_df = df_all[df_all['fold'] == fold].reset_index(drop=True)

    pos_counts = train_df[LABEL_COLS].sum().values
    neg_counts = len(train_df) - pos_counts
    pos_weights = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, dtype=torch.float32).to(CFG.device))

    train_loader = DataLoader(PetDataset(train_df, CFG.train_img_dir, get_transforms('train')),
                              batch_size=CFG.batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(PetDataset(val_df, CFG.train_img_dir, get_transforms('val')),
                            batch_size=CFG.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    model = timm.create_model(CFG.model_name, pretrained=True, num_classes=CFG.num_classes).to(CFG.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scaler = torch.amp.GradScaler('cuda') if CFG.device.type == 'cuda' else None

    best_val_f1 = 0.0
    for epoch in range(CFG.epochs):
        model.train()
        for images, targets in tqdm(train_loader, desc=f"Fold {fold} Ep {epoch+1}"):
            images, targets = images.to(CFG.device), targets.to(CFG.device)
            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(images), targets)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(images), targets)
                loss.backward()
                optimizer.step()

        model.eval()
        val_probs, val_targets = [], []
        with torch.no_grad():
            for images, targets in val_loader:
                val_probs.append(torch.sigmoid(model(images.to(CFG.device))).cpu().numpy())
                val_targets.append(targets.numpy())
        val_probs = np.vstack(val_probs)
        val_targets = np.vstack(val_targets)
        th = find_optimal_thresholds(val_targets, val_probs)
        preds = apply_postprocessing((val_probs >= th).astype(int))
        f1 = f1_score(val_targets, preds, average='macro', zero_division=0)
        print(f"Fold {fold} Epoch {epoch+1} Val Macro F1: {f1:.4f}")
        if f1 > best_val_f1:
            best_val_f1 = f1
            torch.save(model.state_dict(), save_path)

    trained_checkpoints.append(save_path)
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

print("Всі фолди натреновано! Чекпоінти:", trained_checkpoints)


In [ ]:
# 5. Ансамблевий інференс (усереднення ймовірностей усіх моделей)
class TestDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.transform = get_transforms('val')
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(image=img)['image'], row['image_id']

df_test = pd.read_csv(CFG.sample_sub)
test_loader = DataLoader(TestDataset(df_test, CFG.test_img_dir), batch_size=CFG.batch_size, shuffle=False, num_workers=4)

ensemble_probs = np.zeros((len(df_test), CFG.num_classes), dtype=np.float32)
image_ids = df_test['image_id'].tolist()

for ckpt in trained_checkpoints:
    print(f"Інференс для {ckpt}...")
    model = timm.create_model(CFG.model_name, pretrained=False, num_classes=CFG.num_classes).to(CFG.device)
    model.load_state_dict(torch.load(ckpt, map_location=CFG.device))
    model.eval()
    
    ckpt_probs = []
    with torch.no_grad():
        for images, _ in tqdm(test_loader):
            p = torch.sigmoid(model(images.to(CFG.device))).cpu().numpy()
            ckpt_probs.append(p)
    ensemble_probs += np.vstack(ckpt_probs) / len(trained_checkpoints)
    del model
    torch.cuda.empty_cache()

# Застосовуємо пороги до ансамблю
# Підвищені пороги для рідкісних класів відсікають False Positives
FINAL_THRESHOLDS = np.array([0.95, 0.20, 0.85, 0.95, 0.45])
print("Застосовую пороги ансамблю:", FINAL_THRESHOLDS)

final_preds = (ensemble_probs >= FINAL_THRESHOLDS).astype(int)
final_preds = apply_postprocessing(final_preds)

sub = pd.DataFrame(final_preds, columns=LABEL_COLS)
sub.insert(0, 'image_id', image_ids)
out_path = '/content/submission_ensemble_multifold.csv'
sub.to_csv(out_path, index=False)

print(f"\n[✓] ГОТОВО! Ансамблевий файл збережено: {out_path}")
print("Статистика передбачень:")
for col in LABEL_COLS:
    print(f"  {col:7s}: {sub[col].sum():5d} ({sub[col].mean()*100:.2f}%)")

# Автоматичне завантаження на комп'ютер
from google.colab import files
files.download(out_path)
